# 08 · NB-SVM, embeddings e5 y ensambles: el modelo final

El modelo del `06_svd_modelos_stacking.ipynb` (promedio de **LightGBM** y **LightGBM con SVD del texto**) llega a **0,9457 en la validación**. En el `07_embeddings_variantes.ipynb`, otras variantes de modelos sumaban apenas, por debajo del ruido.

Este notebook junta lo que vino después, que es lo que llevó a las submissions 07 a 11:

1. **NB-SVM:** una segunda probabilidad de usado según el texto, calculada de otra forma.
2. **Ensambles con más modelos:** CatBoost, LightGBM DART, XGBoost y LightGBM `extra_trees`, todos con la SVD.
3. **Embeddings e5:** un modelo de lenguaje mejor que el del 07.
4. **Dos ideas descartadas:** palabras exactas como columnas y pseudo-etiquetado de test.
5. **Pesos del ensamble:** promedio simple contra selección *greedy*.
6. **Submissions 07 a 11**, regeneradas acá.

Mismo criterio que siempre: comparación partición por partición; una idea se acepta si mejora más que el ruido (~0,001) y en casi todas las particiones.

**Resultado:** el **modelo final es el ensamble 08**, el promedio simple de 4 modelos (los dos LightGBM con NB-SVM, CatBoost y DART): **0,9462** en la validación y **0,94671** en Kaggle. Es el que reproducen `train.py` y `predict.py`. El ensamble 11 (10 modelos, con embeddings) da apenas más (0,9463 / 0,94690), dentro del ruido y con mucho más costo.

Requiere `torch` y `sentence-transformers`. La primera vez baja el modelo de lenguaje (~470 MB). Tarda en total unas **6 horas**.

## 0. Datos, features, piezas y el modelo 06

Lo mismo que en el 06 y el 07, con una diferencia: las piezas que usan la respuesta se calculan también para la **partición 5** (train → test), porque este notebook genera submissions. Tarda unos 35 minutos.

In [1]:
import time
import warnings
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from features import (cargar_datos, armar_tabla, calcular_piezas, armar_conjunto, texto_combinado, componentes_svd,
                      claves_cruzadas, tabla_target_encoding, aplicar_target_encoding, modelo_texto,
                      matriz_palabras, prob_nbsvm, preparar_catboost, CAT)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

todo, y = cargar_datos()
N_TRAIN = len(y)
X_todo = armar_tabla(todo)
X_svd = pd.concat([X_todo, componentes_svd(todo, 128)], axis=1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
PARTICIONES = list(cv.split(np.zeros(N_TRAIN), y)) + [(np.arange(N_TRAIN), np.arange(N_TRAIN, len(todo)))]

inicio = time.time()
piezas = [calcular_piezas(todo, y, idx_a, idx_b, caracteres=True, cruzados=True) for idx_a, idx_b in PARTICIONES]
print(X_svd.shape, f"piezas listas en {(time.time() - inicio) / 60:.1f} minutos")

(100000, 220) piezas listas en 14.7 minutos


Cada modelo queda descrito por una **especificación** (`ESPEC`): qué features usa, qué algoritmo, con qué hiperparámetros y qué piezas extra. Con la misma especificación se calcula su predicción fuera de fold (particiones 0 a 4, para validar) y su predicción de test (partición 5, para la submission). Así lo que se valida es exactamente lo que se envía.

In [2]:
PARAMS_LGBM = dict(n_estimators=1500, learning_rate=0.02, num_leaves=63, colsample_bytree=0.5, random_state=42, verbose=-1)
PARAMS_DART = {**PARAMS_LGBM, "boosting_type": "dart", "n_estimators": 800, "learning_rate": 0.08, "drop_rate": 0.1}
PARAMS_ET = {**PARAMS_LGBM, "extra_trees": True, "n_estimators": 3000}
PARAMS_XGB = dict(n_estimators=1500, learning_rate=0.02, max_depth=8, subsample=0.8, colsample_bytree=0.5, min_child_weight=2,
                  tree_method="hist", enable_categorical=True, random_state=42, n_jobs=8)
PARAMS_CAT = dict(iterations=2000, learning_rate=0.06, depth=7, random_seed=42, verbose=0, thread_count=8,
                  allow_writing_files=False)

def predecir(f, X, modelo=LGBMClassifier, params=PARAMS_LGBM, extra=None):
    # entrena con las filas de entrenamiento de la partición f y predice las otras (la partición 5 es train → test)
    # extra: piezas adicionales, una entrada por partición
    idx_a, idx_b = PARTICIONES[f]
    A, B = armar_conjunto(X, {**piezas[f], **(extra[f] if extra else {})}, idx_a, idx_b)
    if modelo is CatBoostClassifier:
        A, B = preparar_catboost(A), preparar_catboost(B)
        return modelo(**params).fit(A, y[idx_a], cat_features=CAT).predict_proba(B)[:, 1]
    return modelo(**params).fit(A, y[idx_a]).predict_proba(B)[:, 1]

def oof(X, modelo=LGBMClassifier, params=PARAMS_LGBM, extra=None):
    # probabilidad fuera de fold para los 70.000 ítems de train
    p = np.zeros(N_TRAIN)
    for f in range(5):
        p[PARTICIONES[f][1]] = predecir(f, X, modelo, params, extra)
    return p

def promedio(P, nombres):
    # promedio simple de las probabilidades de varios modelos
    return sum(P[n] for n in nombres) / len(nombres)

def accuracy_por_particion(p):
    return np.array([accuracy_score(y[idx_b], p[idx_b] > 0.5) for _, idx_b in PARTICIONES[:5]])

def desacuerdo(p, q):
    return ((p > 0.5) != (q > 0.5)).mean()

resultados = []
def comparar(nombre, p, ref):
    notas, notas_ref = accuracy_por_particion(p), accuracy_por_particion(ref)
    dif = notas - notas_ref
    resultados.append({"experimento": nombre, "accuracy": round(notas.mean(), 4), "vs. referencia": round(dif.mean(), 4),
                       "mejora en": f"{(dif > 0).sum()}/5", "diferencia por partición": " ".join(f"{d:+.4f}" for d in dif)})
    print(f"{nombre:62s} {notas.mean():.4f}  ({dif.mean():+.4f}, mejora en {(dif > 0).sum()}/5)")

ESPEC = {"lgbm": (X_todo, LGBMClassifier, PARAMS_LGBM, None),
         "lgbm_svd": (X_svd, LGBMClassifier, PARAMS_LGBM, None)}
OOF = {n: oof(*ESPEC[n]) for n in ESPEC}
p_06 = promedio(OOF, ["lgbm", "lgbm_svd"])
print(f"LightGBM {accuracy_por_particion(OOF['lgbm']).mean():.4f} | LightGBM SVD {accuracy_por_particion(OOF['lgbm_svd']).mean():.4f}"
      f" | modelo 06 (promedio) {accuracy_por_particion(p_06).mean():.4f}")

LightGBM 0.9444 | LightGBM SVD 0.9452 | modelo 06 (promedio) 0.9457


## 1. NB-SVM: otra forma de leer el texto

Hasta ahora el texto entra al modelo como **una** probabilidad: TF-IDF (palabras y pedazos de palabras) + regresión logística (`prob_combinado`). NB-SVM (Wang y Manning, 2012) es otra receta clásica para clasificar textos, que combina dos ideas:

1. **De Naive Bayes:** para cada palabra o par de palabras se calcula su **razón de usados**, `r = log(frecuencia entre los usados / frecuencia entre los nuevos)`. "usado" o "impecable" tienen `r` muy positivo; "nuevo", "sellado" o "garantía oficial", muy negativo. Se suma 1 a cada conteo (`alfa = 1`) para que las palabras raras no den valores extremos.
2. **De la regresión logística:** cada texto se representa con **qué palabras tiene** (1 o 0, sin TF-IDF), multiplicadas por su `r`, y sobre eso se ajusta una regresión logística (`C = 0,5`).

Multiplicar por `r` le da a la logística una "pista" de qué palabras importan antes de empezar, y eso la hace más robusta con palabras poco frecuentes. Las funciones están en `features.py` (`matriz_palabras`, `ajustar_nbsvm`, `aplicar_nbsvm`, `prob_nbsvm`).

Como usa la respuesta, `prob_nbsvm` se calcula **fuera de fold**, igual que `prob_combinado`: en cada partición, dentro de las filas de entrenamiento con 5 sub-particiones, y para las filas a predecir, entrenado con todas las de entrenamiento. El vocabulario se arma sobre train + test (no usa la respuesta). Tarda unos 10 minutos.

In [3]:
inicio = time.time()
M = matriz_palabras(todo)
extra_nb = [{"prob_nbsvm": prob_nbsvm(M, y, idx_a, idx_b)} for idx_a, idx_b in PARTICIONES]
print(f"{M.shape[1]:,} palabras y pares de palabras | listo en {(time.time() - inicio) / 60:.1f} minutos")

p_nbsvm_solo, p_tfidf_solo = np.zeros(N_TRAIN), np.zeros(N_TRAIN)
for f, (_, idx_b) in enumerate(PARTICIONES[:5]):
    p_nbsvm_solo[idx_b] = extra_nb[f]["prob_nbsvm"][1]
    p_tfidf_solo[idx_b] = piezas[f]["prob_combinado"][1]
print(f"NB-SVM solo {accuracy_por_particion(p_nbsvm_solo).mean():.4f} | TF-IDF + logística solo {accuracy_por_particion(p_tfidf_solo).mean():.4f}"
      f" | correlación entre las dos {np.corrcoef(p_nbsvm_solo, p_tfidf_solo)[0, 1]:.3f}")

121,378 palabras y pares de palabras | listo en 1.7 minutos
NB-SVM solo 0.8443 | TF-IDF + logística solo 0.8447 | correlación entre las dos 0.951


Las dos probabilidades del texto son parecidas pero no iguales. `prob_nbsvm` entra como **columna extra** en los dos LightGBM del 06. Tarda unos 20 minutos.

In [4]:
ESPEC["lgbm_nb"] = (X_todo, LGBMClassifier, PARAMS_LGBM, extra_nb)
ESPEC["lgbm_svd_nb"] = (X_svd, LGBMClassifier, PARAMS_LGBM, extra_nb)
for n in ["lgbm_nb", "lgbm_svd_nb"]:
    OOF[n] = oof(*ESPEC[n])

comparar("1. LightGBM + prob_nbsvm (vs. LightGBM)", OOF["lgbm_nb"], OOF["lgbm"])
comparar("1. LightGBM SVD + prob_nbsvm (vs. LightGBM SVD)", OOF["lgbm_svd_nb"], OOF["lgbm_svd"])
comparar("1. par con NB-SVM = submission 07 (vs. modelo 06)", promedio(OOF, ["lgbm_nb", "lgbm_svd_nb"]), p_06)

1. LightGBM + prob_nbsvm (vs. LightGBM)                        0.9447  (+0.0003, mejora en 4/5)
1. LightGBM SVD + prob_nbsvm (vs. LightGBM SVD)                0.9453  (+0.0001, mejora en 2/5)
1. par con NB-SVM = submission 07 (vs. modelo 06)              0.9460  (+0.0003, mejora en 5/5)


**Lectura:** NB-SVM solo es tan bueno como el TF-IDF (0,844 contra 0,845), pero no son iguales (correlación 0,95): se equivocan en ítems algo distintos. Como columna extra, a cada LightGBM por separado lo mejora poco y no siempre (4/5 y 2/5), pero al **promedio de los dos lo mejora en las 5 particiones**. Es poco (+0,0003), pero consistente: es la **submission 07**.

## 2. Ensambles con más modelos

En el 07, CatBoost y LightGBM DART con la SVD sumaban algo como tercer o cuarto modelo, aunque por debajo del ruido. Acá se prueban de nuevo, ahora junto al par con NB-SVM, y se agregan XGBoost y LightGBM `extra_trees` (también con la SVD):

- **CatBoost:** árboles simétricos (todos los nodos de un nivel cortan por la misma columna). Aprende distinto que LightGBM.
- **LightGBM DART:** durante el entrenamiento "apaga" al azar árboles ya construidos, para que los nuevos no dependan tanto de los primeros.
- **XGBoost:** árboles por profundidad (`max_depth=8`) en vez de por hojas.
- **LightGBM `extra_trees`:** cortes al azar, árboles más distintos entre sí.

Estos cuatro **no** llevan `prob_nbsvm`: así se equivocan en ítems todavía más distintos que los dos LightGBM. Tarda unas 2 horas (CatBoost solo, más de 1,5).

In [5]:
ESPEC["dart_svd"] = (X_svd, LGBMClassifier, PARAMS_DART, None)
ESPEC["et_svd"] = (X_svd, LGBMClassifier, PARAMS_ET, None)
ESPEC["xgb_svd"] = (X_svd, XGBClassifier, PARAMS_XGB, None)
ESPEC["cat_svd"] = (X_svd, CatBoostClassifier, PARAMS_CAT, None)
for n in ["dart_svd", "et_svd", "xgb_svd", "cat_svd"]:
    inicio = time.time()
    OOF[n] = oof(*ESPEC[n])
    comparar(f"2. {n} solo, {(time.time() - inicio) / 60:.0f} min (vs. LightGBM SVD)", OOF[n], OOF["lgbm_svd"])
    print(f"   no coincide con LightGBM SVD en el {desacuerdo(OOF[n], OOF['lgbm_svd']):.1%}")

ENSAMBLES = {
    "modelo 06 + CatBoost + DART": ["lgbm", "lgbm_svd", "cat_svd", "dart_svd"],
    "par NB-SVM + CatBoost": ["lgbm_nb", "lgbm_svd_nb", "cat_svd"],
    "par NB-SVM + CatBoost + DART = submission 08": ["lgbm_nb", "lgbm_svd_nb", "cat_svd", "dart_svd"],
    "modelo 06 + par NB-SVM + CatBoost + DART": ["lgbm", "lgbm_svd", "lgbm_nb", "lgbm_svd_nb", "cat_svd", "dart_svd"],
    "los 8 de boosting = submission 09": ["lgbm", "lgbm_svd", "lgbm_nb", "lgbm_svd_nb", "cat_svd", "dart_svd", "xgb_svd", "et_svd"],
}
for nombre, modelos in ENSAMBLES.items():
    comparar(f"2. {nombre} (vs. modelo 06)", promedio(OOF, modelos), p_06)

2. dart_svd solo, 7 min (vs. LightGBM SVD)                     0.9445  (-0.0007, mejora en 1/5)
   no coincide con LightGBM SVD en el 1.0%


2. et_svd solo, 6 min (vs. LightGBM SVD)                       0.9450  (-0.0002, mejora en 2/5)
   no coincide con LightGBM SVD en el 1.2%


2. xgb_svd solo, 10 min (vs. LightGBM SVD)                     0.9447  (-0.0005, mejora en 3/5)
   no coincide con LightGBM SVD en el 0.8%


2. cat_svd solo, 98 min (vs. LightGBM SVD)                     0.9443  (-0.0009, mejora en 1/5)
   no coincide con LightGBM SVD en el 1.4%
2. modelo 06 + CatBoost + DART (vs. modelo 06)                 0.9460  (+0.0003, mejora en 4/5)
2. par NB-SVM + CatBoost (vs. modelo 06)                       0.9461  (+0.0004, mejora en 4/5)
2. par NB-SVM + CatBoost + DART = submission 08 (vs. modelo 06) 0.9462  (+0.0005, mejora en 4/5)
2. modelo 06 + par NB-SVM + CatBoost + DART (vs. modelo 06)    0.9461  (+0.0004, mejora en 3/5)


2. los 8 de boosting = submission 09 (vs. modelo 06)           0.9460  (+0.0003, mejora en 5/5)


**Lectura:** cada modelo nuevo, solo, es igual o peor que LightGBM con SVD; lo que suma es **promediarlos**, porque cada uno se equivoca en ítems algo distintos.

El mejor es el **ensamble 08**: el par con NB-SVM + CatBoost + DART, **0,9462** (+0,0005 sobre el 06). Agregar más modelos (los 8 de la submission 09) no mejora: con modelos tan parecidos, después de 4 se llega a una meseta.

El 08 es el **modelo final**: el mejor de la validación con pocos modelos, todos de boosting sobre las mismas features, y cada parte se puede explicar.

## 3. Embeddings e5

En el 07, los embeddings de `paraphrase-multilingual-MiniLM-L12-v2` (un modelo entrenado para detectar si dos frases dicen lo mismo) daban solo 0,743 y no sumaban.

Acá se prueba `intfloat/multilingual-e5-small`: mismo tamaño (384 números por texto), pero entrenado para **búsqueda** (encontrar el texto que responde a una consulta), que se parece más a lo que se necesita: textos con detalles concretos. Se le pone el prefijo `"query: "`, que es como lo pide el modelo. No usa la respuesta: se calcula una vez para train + test. Tarda alrededor de 1 hora.

In [6]:
from sentence_transformers import SentenceTransformer

inicio = time.time()
lenguaje = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu")
lenguaje.max_seq_length = 96
E = lenguaje.encode(("query: " + texto_combinado(todo)).tolist(), batch_size=128, normalize_embeddings=True,
                    show_progress_bar=False).astype(np.float32)
print(E.shape, f"listo en {(time.time() - inicio) / 60:.1f} minutos")

C:\Users\Ezequiel\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  15%|█▍        | 29/199 [00:00<00:00, 284.47it/s]

Loading weights:  29%|██▉       | 58/199 [00:00<00:00, 279.30it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 799.68it/s]

(100000, 384) listo en 57.6 minutos


Las mismas dos piezas que en el 07, calculadas fuera de fold:

- **`prob_emb`:** regresión logística sobre los 384 números → probabilidad de usado.
- **Vecinos por significado:** las 10 publicaciones más parecidas de **otros vendedores**, qué proporción de ellas era usada (`emb_knn_media`, ponderada por el parecido) y qué tan parecida es la más cercana (`emb_knn_sim`). Se excluye al mismo vendedor porque su historial ya lo mide el target encoding.

Tarda unos 15 minutos.

In [7]:
vendedor = todo["seller_id"].values
sub_cv = lambda: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def prob_emb(idx_a, idx_b):
    pa = np.zeros(len(idx_a))
    for a2, b2 in sub_cv().split(idx_a, y[idx_a]):
        pa[b2] = LogisticRegression(C=2, max_iter=3000).fit(E[idx_a[a2]], y[idx_a[a2]]).predict_proba(E[idx_a[b2]])[:, 1]
    return pa, LogisticRegression(C=2, max_iter=3000).fit(E[idx_a], y[idx_a]).predict_proba(E[idx_b])[:, 1]

def vecinos(idx_ref, idx_obj, K=10):
    salida, E_ref, y_ref, v_ref = np.zeros((len(idx_obj), 2), np.float32), E[idx_ref].T, y[idx_ref], vendedor[idx_ref]
    for i in range(0, len(idx_obj), 2000):
        sl = slice(i, i + 2000)
        sim = E[idx_obj[sl]] @ E_ref                                     # similitud coseno (los vectores tienen norma 1)
        sim[vendedor[idx_obj[sl]][:, None] == v_ref[None, :]] = -1       # fuera el mismo vendedor
        top = np.argpartition(-sim, K, axis=1)[:, :K]
        s = np.take_along_axis(sim, top, 1)
        w = np.clip(s, 0, None)
        salida[sl, 0] = (w * y_ref[top]).sum(1) / np.maximum(w.sum(1), 1e-6)
        salida[sl, 1] = s.max(1)
    return salida

inicio = time.time()
extra_e5 = []
for idx_a, idx_b in PARTICIONES:
    fa = np.zeros((len(idx_a), 2), np.float32)
    for a2, b2 in sub_cv().split(idx_a, y[idx_a]):
        fa[b2] = vecinos(idx_a[a2], idx_a[b2])
    fb = vecinos(idx_a, idx_b)
    extra_e5.append({"prob_emb": prob_emb(idx_a, idx_b), "emb_knn_media": (fa[:, 0], fb[:, 0]), "emb_knn_sim": (fa[:, 1], fb[:, 1])})
print(f"piezas de embeddings en {(time.time() - inicio) / 60:.1f} minutos")

p_emb_solo = np.zeros(N_TRAIN)
for f, (_, idx_b) in enumerate(PARTICIONES[:5]):
    p_emb_solo[idx_b] = extra_e5[f]["prob_emb"][1]
print(f"prob_emb sola: {accuracy_por_particion(p_emb_solo).mean():.4f}")

piezas de embeddings en 12.2 minutos
prob_emb sola: 0.7723


Se prueban como columnas extra en los dos LightGBM: primero solas, después junto a `prob_nbsvm`. Tarda unos 40 minutos.

In [8]:
extra_nb_e5 = [{**extra_nb[f], **extra_e5[f]} for f in range(len(PARTICIONES))]

p_lgbm_e5 = oof(X_todo, extra=extra_e5)
p_lgbm_svd_e5 = oof(X_svd, extra=extra_e5)
comparar("3. LightGBM + e5 (vs. LightGBM)", p_lgbm_e5, OOF["lgbm"])
comparar("3. par con e5 (vs. modelo 06)", (p_lgbm_e5 + p_lgbm_svd_e5) / 2, p_06)

ESPEC["lgbm_nb_e5"] = (X_todo, LGBMClassifier, PARAMS_LGBM, extra_nb_e5)
ESPEC["lgbm_svd_nb_e5"] = (X_svd, LGBMClassifier, PARAMS_LGBM, extra_nb_e5)
for n in ["lgbm_nb_e5", "lgbm_svd_nb_e5"]:
    OOF[n] = oof(*ESPEC[n])
p_07 = promedio(OOF, ["lgbm_nb", "lgbm_svd_nb"])
p_08 = promedio(OOF, ENSAMBLES["par NB-SVM + CatBoost + DART = submission 08"])
comparar("3. par con NB-SVM + e5 (vs. par con NB-SVM)", promedio(OOF, ["lgbm_nb_e5", "lgbm_svd_nb_e5"]), p_07)

ENSAMBLES["par NB-SVM + e5, par NB-SVM, CatBoost, DART = submission 10"] = [
    "lgbm_nb_e5", "lgbm_svd_nb_e5", "lgbm_nb", "lgbm_svd_nb", "cat_svd", "dart_svd"]
ENSAMBLES["los 10 modelos = submission 11"] = list(ESPEC)
for nombre in list(ENSAMBLES)[-2:]:
    comparar(f"3. {nombre} (vs. 08)", promedio(OOF, ENSAMBLES[nombre]), p_08)

3. LightGBM + e5 (vs. LightGBM)                                0.9449  (+0.0005, mejora en 5/5)
3. par con e5 (vs. modelo 06)                                  0.9460  (+0.0003, mejora en 3/5)


3. par con NB-SVM + e5 (vs. par con NB-SVM)                    0.9459  (-0.0001, mejora en 1/5)
3. par NB-SVM + e5, par NB-SVM, CatBoost, DART = submission 10 (vs. 08) 0.9461  (-0.0000, mejora en 1/5)
3. los 10 modelos = submission 11 (vs. 08)                     0.9463  (+0.0001, mejora en 1/5)


**Lectura:** e5 es bastante mejor que MiniLM (0,77 contra 0,74 sola), y como columnas extra en LightGBM suma un poco en las 5 particiones. Pero **junto con NB-SVM ya no suma**: las dos piezas aportan casi la misma información nueva sobre el texto.

Los ensambles que agregan los modelos con e5 (submissions 10 y 11) quedan **igual que el 08**, a ±0,0001: dentro del ruido. No justifican agregar un modelo de lenguaje al modelo final.

## 4. Dos ideas descartadas

### 4.1 Palabras exactas como columnas

En vez de resumir el texto en una probabilidad o en la SVD, se le dan a LightGBM directamente las **300 palabras o pares de palabras** que más distinguen nuevo de usado (test chi², elegidas **solo con las filas de entrenamiento** de cada partición), como columnas 0/1. Tarda unos 15 minutos.

In [9]:
vect = CountVectorizer(ngram_range=(1, 2), min_df=20, binary=True, dtype=np.float32)
M20 = vect.fit_transform(texto_combinado(todo)).tocsr()
vocabulario = np.array(vect.get_feature_names_out())

p_palabras = np.zeros(N_TRAIN)
for f in range(5):
    idx_a, idx_b = PARTICIONES[f]
    top = np.argsort(-np.nan_to_num(chi2(M20[idx_a], y[idx_a])[0]))[:300]
    W = pd.DataFrame(M20[:, top].toarray(), columns=[f"w_{j}" for j in range(300)])   # LightGBM no acepta tildes en los nombres
    p_palabras[idx_b] = predecir(f, pd.concat([X_todo, W], axis=1))
    if f == 0:
        print("las que más distinguen:", list(vocabulario[top[:15]]))
comparar("4.1 LightGBM + 300 palabras (vs. LightGBM)", p_palabras, OOF["lgbm"])
comparar("4.1 modelo 06 + LightGBM con 300 palabras (vs. modelo 06)", (OOF["lgbm"] + OOF["lgbm_svd"] + p_palabras) / 3, p_06)

las que más distinguen: ['meses', 'garantia', 'nuevo', 'estado', 'por', 'part_number', 'mla', 'mla part_number', 'revista', 'vinilo', 'kit', 'antiguo', 'antigua', 'oficial', 'horpref']


4.1 LightGBM + 300 palabras (vs. LightGBM)                     0.9444  (+0.0001, mejora en 2/5)
4.1 modelo 06 + LightGBM con 300 palabras (vs. modelo 06)      0.9456  (-0.0001, mejora en 2/5)


**Lectura:** no suma. Las palabras más útiles ya están en las probabilidades del texto y en las columnas a mano del 02 (`titulo_usad`, `titulo_nuev`...).

### 4.2 Pseudo-etiquetado

Test tiene 30.000 ítems sin respuesta, y el 72% son de vendedores que también están en train. Idea: usar las predicciones **muy seguras** de test (probabilidad > 0,97 o < 0,03) como si fueran respuestas, para que los target encodings de vendedor tengan más datos.

Para validarlo sin hacer trampa, en cada partición las pseudo-etiquetas de test se sacan del modelo 06 **entrenado solo con las filas de entrenamiento de esa partición**. Después se recalculan los 5 target encodings con esas filas + las de test seguras, y se entrena el par del 06. Tarda unos 50 minutos.

In [10]:
IDX_TEST = np.arange(N_TRAIN, len(todo))
texto = texto_combinado(todo)
sub = lambda s, idx: s.iloc[idx].reset_index(drop=True)
CLAVES = {"te_seller_id": todo["seller_id"], "te_category_id": todo["category_id"], **claves_cruzadas(todo)}

# 1. en cada partición, el modelo 06 entrenado con sus filas de entrenamiento predice test
inicio = time.time()
p_test_particion = []
for f in range(5):
    idx_a, _ = PARTICIONES[f]
    prior = y[idx_a].mean()
    pz = {}
    for n, (pa, _) in piezas[f].items():   # en train, las mismas piezas fuera de fold; en test, ajustadas con idx_a
        if n == "prob_combinado":
            pb = modelo_texto(caracteres=True).fit(sub(texto, idx_a), y[idx_a]).predict_proba(sub(texto, IDX_TEST))[:, 1]
        else:
            pb = aplicar_target_encoding(tabla_target_encoding(sub(CLAVES[n], idx_a), y[idx_a], prior), prior, sub(CLAVES[n], IDX_TEST))
        pz[n] = (pa, pb)
    q = 0
    for X in [X_todo, X_svd]:
        A, B = armar_conjunto(X, pz, idx_a, IDX_TEST)
        q = q + LGBMClassifier(**PARAMS_LGBM).fit(A, y[idx_a]).predict_proba(B)[:, 1] / 2
    p_test_particion.append(q)
print(f"test predicho en cada partición: {(time.time() - inicio) / 60:.1f} minutos")

# 2. target encoding con las respuestas de entrenamiento + las pseudo-etiquetas de test
def te_con_pseudo(clave, idx_a, idx_b, idx_ps, y_ps, suavizado=10):
    prior = y[idx_a].mean()
    ca, cb, cp = sub(clave, idx_a), sub(clave, idx_b), sub(clave, idx_ps)
    enc_a = np.full(len(idx_a), prior)
    for a2, b2 in sub_cv().split(idx_a, y[idx_a]):
        tabla = tabla_target_encoding(pd.concat([ca.iloc[a2], cp]), np.concatenate([y[idx_a][a2], y_ps]), prior, suavizado)
        enc_a[b2] = aplicar_target_encoding(tabla, prior, ca.iloc[b2])
    tabla = tabla_target_encoding(pd.concat([ca, cp]), np.concatenate([y[idx_a], y_ps]), prior, suavizado)
    return enc_a, aplicar_target_encoding(tabla, prior, cb)

UMBRAL = 0.97
p_pseudo = np.zeros(N_TRAIN)
for f in range(5):
    idx_a, idx_b = PARTICIONES[f]
    seguro = (p_test_particion[f] > UMBRAL) | (p_test_particion[f] < 1 - UMBRAL)
    idx_ps, y_ps = IDX_TEST[seguro], (p_test_particion[f][seguro] > 0.5).astype(int)
    pz = {n: (te_con_pseudo(CLAVES[n], idx_a, idx_b, idx_ps, y_ps) if n in CLAVES else v) for n, v in piezas[f].items()}
    q = 0
    for X in [X_todo, X_svd]:
        A, B = armar_conjunto(X, pz, idx_a, idx_b)
        q = q + LGBMClassifier(**PARAMS_LGBM).fit(A, y[idx_a]).predict_proba(B)[:, 1] / 2
    p_pseudo[idx_b] = q
    if f == 0:
        print(f"{seguro.mean():.0%} de test recibe pseudo-etiqueta")
comparar(f"4.2 modelo 06 con pseudo-etiquetas de test, umbral {UMBRAL} (vs. modelo 06)", p_pseudo, p_06)

test predicho en cada partición: 8.1 minutos


76% de test recibe pseudo-etiqueta


4.2 modelo 06 con pseudo-etiquetas de test, umbral 0.97 (vs. modelo 06) 0.9448  (-0.0009, mejora en 0/5)


**Lectura:** el pseudo-etiquetado **empeora en las 5 particiones** (−0,0009). La explicación más probable: las predicciones seguras son de vendedores que el modelo ya conoce bien, así que no agregan información nueva; y los errores seguros (el ruido de etiqueta que se vio en el 05) entran como si fueran respuestas y se refuerzan.

## 5. Pesos del ensamble: promedio simple o selección *greedy*

Hasta acá los ensambles son **promedios simples** (todos los modelos pesan igual). Una alternativa conocida es la **selección greedy** (Caruana, 2004): se arranca con el mejor modelo y en cada paso se agrega (con reemplazo) el que más mejora la accuracy del promedio. Un modelo elegido 3 veces pesa el triple.

El problema es que los pesos se eligen **mirando la misma validación** con la que se los evalúa. Para medirlo bien se hace una evaluación **anidada**: los pesos se eligen con 4 particiones y se evalúan en la 5ª, rotando.

In [11]:
# mismos 10 modelos, en el orden en que se probaron originalmente (importa para desempatar)
O = {n: OOF[n] for n in ["lgbm", "lgbm_svd", "lgbm_nb", "lgbm_svd_nb", "lgbm_nb_e5", "lgbm_svd_nb_e5",
                         "cat_svd", "dart_svd", "xgb_svd", "et_svd"]}

def greedy(puntaje, pasos):
    elegidos, suma = [], np.zeros(N_TRAIN)
    for _ in range(pasos):
        mejor = max(O, key=lambda n: puntaje((suma + O[n]) / (len(elegidos) + 1)))
        elegidos.append(mejor)
        suma = suma + O[mejor]
    return suma / len(elegidos), Counter(elegidos)

# a) pesos elegidos y evaluados con las mismas 5 particiones
p_greedy, pesos = greedy(lambda q: accuracy_por_particion(q).mean(), 12)
print("pesos elegidos:", dict(pesos))
comparar("5. greedy, 12 pasos, elegido y evaluado con lo mismo (vs. 08)", p_greedy, p_08)

# b) anidado: pesos elegidos sin mirar la partición que se evalúa
for pasos in [4, 8, 12]:
    p_anidado = np.zeros(N_TRAIN)
    for f in range(5):
        dentro = np.concatenate([PARTICIONES[g][1] for g in range(5) if g != f])
        q, _ = greedy(lambda q: accuracy_score(y[dentro], q[dentro] > 0.5), pasos)
        p_anidado[PARTICIONES[f][1]] = q[PARTICIONES[f][1]]
    comparar(f"5. greedy anidado, {pasos} pasos (vs. 08)", p_anidado, p_08)

pesos elegidos: {'lgbm_svd_nb_e5': 1, 'lgbm_nb': 3, 'cat_svd': 3, 'et_svd': 1, 'lgbm_nb_e5': 2, 'dart_svd': 1, 'lgbm_svd': 1}
5. greedy, 12 pasos, elegido y evaluado con lo mismo (vs. 08)  0.9467  (+0.0005, mejora en 3/5)


5. greedy anidado, 4 pasos (vs. 08)                            0.9458  (-0.0003, mejora en 2/5)


5. greedy anidado, 8 pasos (vs. 08)                            0.9459  (-0.0002, mejora en 2/5)


5. greedy anidado, 12 pasos (vs. 08)                           0.9459  (-0.0003, mejora en 2/5)


**Lectura:** elegido y evaluado con lo mismo, el greedy parece el mejor de todos. Pero evaluado de forma honesta (anidada) queda **por debajo del 08**: los pesos se ajustan al azar de estas particiones (**sobreajuste a la validación**), no a algo que se repita en datos nuevos. Con modelos tan parecidos, el promedio simple es más robusto. Por eso todas las submissions son promedios simples.

## 6. Submissions 07 a 11

Cada modelo se entrena con los 70.000 ítems de train (partición 5) y predice test. Cada submission es el promedio simple de sus modelos, con umbral 0,5. Se comparan con los archivos que se enviaron a Kaggle. Tarda unos 40 minutos.

In [12]:
inicio = time.time()
P_TEST = {n: predecir(5, *ESPEC[n]) for n in ESPEC}
print(f"10 modelos entrenados con todo train en {(time.time() - inicio) / 60:.1f} minutos")

SUBMISSIONS = {
    "07_par_nbsvm": ["lgbm_nb", "lgbm_svd_nb"],
    "08_ensamble_4_nbsvm_cat_dart": ENSAMBLES["par NB-SVM + CatBoost + DART = submission 08"],
    "09_ensamble_8_boosting": ENSAMBLES["los 8 de boosting = submission 09"],
    "10_ensamble_6_e5": ENSAMBLES["par NB-SVM + e5, par NB-SVM, CatBoost, DART = submission 10"],
    "11_ensamble_10_todos": ENSAMBLES["los 10 modelos = submission 11"],
}
KAGGLE = {"07_par_nbsvm": 0.94600, "08_ensamble_4_nbsvm_cat_dart": 0.94671, "09_ensamble_8_boosting": 0.94666,
          "10_ensamble_6_e5": 0.94666, "11_ensamble_10_todos": 0.94690}
tabla = []
for nombre, modelos in SUBMISSIONS.items():
    p = promedio(P_TEST, modelos)
    s = pd.DataFrame({"ID": range(1, len(p) + 1), "condition": np.where(p > 0.5, "used", "new")})
    enviada = pd.read_csv(f"submissions/{nombre}.csv")
    s.to_csv(f"submissions/{nombre}.csv", index=False)
    tabla.append({"submission": nombre, "modelos": len(modelos),
                  "accuracy validación": round(accuracy_por_particion(promedio(OOF, modelos)).mean(), 4),
                  "Kaggle (público)": KAGGLE[nombre],
                  "predicciones distintas a la enviada": int((s["condition"] != enviada["condition"]).sum())})
pd.DataFrame(tabla)

10 modelos entrenados con todo train en 36.1 minutos


,submission,modelos,accuracy validación,Kaggle (público),predicciones distintas a la enviada
0,07_par_nbsvm,2,0.9460,0.94600,0
1,08_ensamble_4_nbsvm_cat_dart,4,0.9462,0.94671,0
2,09_ensamble_8_boosting,8,0.9460,0.94666,0
3,10_ensamble_6_e5,6,0.9461,0.94666,0
4,11_ensamble_10_todos,10,0.9463,0.94690,0


## 7. Resumen

In [13]:
pd.DataFrame(resultados)

,experimento,accuracy,vs. referencia,mejora en,diferencia por partición
0,1. LightGBM + prob_nbsvm (vs. LightGBM),0.9447,0.0003,4/5,+0.0005 +0.0012 -0.0014 +0.0010 +0.0004
1,1. LightGBM SVD + prob_nbsvm (vs. LightGBM SVD),0.9453,0.0001,2/5,-0.0005 -0.0002 +0.0000 +0.0006 +0.0006
2,1. par con NB-SVM = submission 07 (vs. modelo 06),0.9460,0.0003,5/5,+0.0001 +0.0004 +0.0001 +0.0006 +0.0002
3,"2. dart_svd solo, 7 min (vs. LightGBM SVD)",0.9445,-0.0007,1/5,-0.0006 -0.0007 -0.0006 +0.0001 -0.0016
4,"2. et_svd solo, 6 min (vs. LightGBM SVD)",0.9450,-0.0002,2/5,+0.0006 -0.0010 -0.0009 +0.0015 -0.0009
5,"2. xgb_svd solo, 10 min (vs. LightGBM SVD)",0.9447,-0.0005,3/5,-0.0010 -0.0018 +0.0001 +0.0001 +0.0001
6,"2. cat_svd solo, 98 min (vs. LightGBM SVD)",0.9443,-0.0009,1/5,-0.0003 -0.0019 +0.0009 -0.0009 -0.0024
7,2. modelo 06 + CatBoost + DART (vs. modelo 06),0.9460,0.0003,4/5,+0.0006 +0.0004 +0.0007 -0.0001 +0.0002
8,2. par NB-SVM + CatBoost (vs. modelo 06),0.9461,0.0004,4/5,+0.0004 +0.0005 +0.0010 +0.0000 +0.0001
9,2. par NB-SVM + CatBoost + DART = submission 0...,0.9462,0.0005,4/5,-0.0004 +0.0009 +0.0011 +0.0006 +0.0001


- **NB-SVM** mejora al par del 06 en las 5 particiones: una segunda probabilidad del texto que se equivoca distinto que la del TF-IDF.
- **Promediar modelos distintos** (CatBoost y DART con la SVD) suma un poco más, aunque solos sean peores. Con 4 modelos se llega a la meseta: con 8 o 10 no mejora.
- Los **embeddings e5** suman solos, pero no encima de NB-SVM. El pseudo-etiquetado empeora y las palabras exactas no suman.
- Elegir pesos con la validación la **sobreajusta**: el promedio simple es más robusto.

**Modelo final: ensamble 08** (0,9462 en validación, 0,94671 en Kaggle): el promedio simple de
1. LightGBM con las 98 columnas del 05 + `prob_nbsvm`;
2. lo mismo + las 128 columnas de la SVD;
3. CatBoost con las 98 columnas + SVD;
4. LightGBM DART con las 98 columnas + SVD.

Lo entrenan `train.py` y `predict.py`, que regeneran exactamente `submissions/08_ensamble_4_nbsvm_cat_dart.csv`. La 11 (0,94690 en Kaggle) se eligió como segunda submission para el ranking privado: la diferencia con la 08 está dentro del ruido del leaderboard (~±0,002).